# 07c | Drills: Linear regression

Goes with **07a and 07b**: the `statsmodels` formula interface, extracting coefficients and intervals, OLS by hand, R², prediction, interaction terms, log transforms, robust standard errors, VIF, omitted variable bias, and the usual follow-up questions about OLS assumptions.

**How to use this page:** There are no input boxes here. There are two kinds of problems:

- **Computation problems**: the code and its output are shown right below. First think about which function and formula to use, then read on for the answer;
- **Spoken problems** (marked 🗣 after the title): answer out loud first (30–60 seconds), then compare with the reference answer.

Problems on a page run in order, so variables and functions defined earlier can be reused later. Every problem assumes this has already been run:

```python
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

rng = np.random.default_rng(7)
n = 500
df = pd.DataFrame({"x1": rng.normal(size=n), "x2": rng.normal(size=n),
                   "group": rng.choice(["a", "b", "c"], size=n)})
df["y"] = (2 + 1.5 * df["x1"] - 0.8 * df["x2"]
           + np.where(df["group"] == "b", 1.0, 0.0) + rng.normal(scale=1.5, size=n))
```

### Problem 1 · Fit and look at the coefficients

Fit `y ~ x1 + x2 + C(group)` and look at all the coefficients.

```python
model = smf.ols("y ~ x1 + x2 + C(group)", data=df).fit()
model.params.round(3)
```

**Result:**

|   |   |
|---|---|
| Intercept | 1.977 |
| C(group)[T.b] | 1.101 |
| C(group)[T.c] | -0.059 |
| x1 | 1.503 |
| x2 | -0.762 |

*5 rows × 1 column*

> True values: intercept 2, x1 is 1.5, x2 is −0.8; group b is 1 higher than group a, and group c is the same as a. `C(group)[T.b]` is the difference between b and the reference category a.

### Problem 2 · Everything about one coefficient

The coefficient on `x1`, its standard error, p-value and 95% confidence interval.

```python
ci = model.conf_int()
print(round(model.params["x1"], 4), round(model.bse["x1"], 4), f"{model.pvalues['x1']:.1e}")
print(round(ci.loc["x1", 0], 4), round(ci.loc["x1", 1], 4))
```

**Result:**

```text
1.5028 0.0762 2.7e-64
1.353 1.6525
```

> R's `coef`, `summary` and `confint` correspond to `.params`, `.bse` / `.pvalues`, and `.conf_int()`.

### Problem 3 · OLS by hand

Without statsmodels, compute the coefficients of `y ~ x1 + x2` with the normal equations, then check against statsmodels.

```python
X = np.column_stack([np.ones(n), df["x1"], df["x2"]])
y = df["y"].to_numpy()
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(beta.round(4))
print(smf.ols("y ~ x1 + x2", data=df).fit().params.round(4).to_numpy())
```

**Result:**

```text
[ 2.3121  1.5225 -0.7771]
[ 2.3121  1.5225 -0.7771]
```

> Add a column of ones yourself for the intercept; solve the system with `solve` rather than an explicit inverse. The intercept is about 2.3 rather than 2: this model leaves out `group`, so part of group b's extra 1 is absorbed by the intercept.

### Problem 4 · R² and adjusted R² by hand

Use `beta` from the previous problem to compute R² and adjusted R², and check against statsmodels.

```python
y_hat = X @ beta
r2 = 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()
p = 2                                          # number of predictors (excluding the intercept)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
m12 = smf.ols("y ~ x1 + x2", data=df).fit()
print(round(r2, 4), round(r2_adj, 4))
print(round(m12.rsquared, 4), round(m12.rsquared_adj, 4))
```

**Result:**

```text
0.4792 0.4771
0.4792 0.4771
```

> Adjusted R² penalizes extra variables through degrees of freedom: $1 - (1 - R^2)\frac{n-1}{n-p-1}$.

### Problem 5 · Predicting new data

Use `model` from Problem 1 to predict two new users: (x1 = 1, x2 = 0, group a) and (x1 = 1, x2 = 0, group b).

```python
new = pd.DataFrame({"x1": [1.0, 1.0], "x2": [0.0, 0.0], "group": ["a", "b"]})
model.predict(new).round(3)
```

**Result:**

|   |   |
|---|---|
| 0 | 3.48 |
| 1 | 4.581 |

*2 rows × 1 column*

> The new data's column names must match those in the formula, and categorical values must have appeared in the training data. The difference between the two predictions is the `C(group)[T.b]` coefficient.

### Problem 6 · Interaction terms

The effect of `x` on `y` varies with `z`. Fit `y ~ x * z` and compute the marginal effect of x at z = 0 and z = 1.

```python
rng = np.random.default_rng(8)
d2 = pd.DataFrame({"x": rng.normal(size=n), "z": rng.normal(size=n)})
d2["y"] = 1 + 1.0 * d2["x"] + 0.5 * d2["z"] + 0.5 * d2["x"] * d2["z"] + rng.normal(size=n)

m_int = smf.ols("y ~ x * z", data=d2).fit()
b = m_int.params
print("effect of x at z = 0:", round(b["x"], 3))
print("effect of x at z = 1:", round(b["x"] + b["x:z"] * 1, 3))
b.round(3)
```

**Result:**

```text
effect of x at z = 0: 0.963
effect of x at z = 1: 1.446
```

|   |   |
|---|---|
| Intercept | 0.916 |
| x | 0.963 |
| z | 0.464 |
| x:z | 0.483 |

*4 rows × 1 column*

> True values: intercept 1, x is 1.0, z is 0.5, interaction 0.5. The marginal effect of x is $\beta_x + \beta_{xz} z$. With an interaction term, the coefficient on `x` by itself is only the effect at z = 0.

### Problem 7 · Log outcome

The log of income y is linear in x. Fit `np.log(y) ~ x` and interpret the coefficient on x.

```python
rng = np.random.default_rng(9)
d3 = pd.DataFrame({"x": rng.normal(5, 2, n)})
d3["y"] = np.exp(1 + 0.1 * d3["x"] + rng.normal(scale=0.3, size=n))

b_log = smf.ols("np.log(y) ~ x", data=d3).fit().params["x"]
print(round(b_log, 4), round(np.exp(b_log) - 1, 4))
```

**Result:**

```text
0.0987 0.1038
```

> A 1-unit increase in x changes y by about 100·β%, roughly 10%; the exact change is $e^\beta - 1$. For small β the two are nearly identical.

### Problem 8 · Robust standard errors

Data whose error grows with x: compare the classical standard error with the HC3 robust standard error.

```python
rng = np.random.default_rng(10)
h = pd.DataFrame({"x": rng.uniform(0, 10, 400)})
h["y"] = 1 + 0.8 * h["x"] + rng.normal(scale=0.2 * h["x"] ** 1.5)

classic = smf.ols("y ~ x", data=h).fit()
robust = smf.ols("y ~ x", data=h).fit(cov_type="HC3")
print(round(classic.params["x"], 4), round(robust.params["x"], 4))
print(round(classic.bse["x"], 4), round(robust.bse["x"], 4))
```

**Result:**

```text
0.8143 0.8143
0.0564 0.0663
```

> The coefficients are identical; only the standard errors change. Here the classical standard error is too small and would make you overconfident.

### Problem 9 · VIF

Construct an x3 that is highly correlated with x1 and x2, and compute the VIF of all three variables.

```python
rng = np.random.default_rng(11)
c = pd.DataFrame({"x1": rng.normal(size=n), "x2": rng.normal(size=n)})
c["x3"] = c["x1"] + 0.5 * c["x2"] + rng.normal(scale=0.2, size=n)

Xc = np.column_stack([np.ones(n), c["x1"], c["x2"], c["x3"]])
[round(float(variance_inflation_factor(Xc, j)), 1) for j in [1, 2, 3]]
```

**Result:**

```text
[24.2, 7.6, 31.3]
```

> Column 0 is the intercept and is skipped. x3 is almost a linear combination of x1 and x2: the VIFs of x1 and x3 exceed 20, and x2's exceeds 5 (the usual warning thresholds are 5 or 10). Equivalent computation: regress x3 on x1 and x2 to get R², then VIF = 1/(1 − R²).

### Problem 10 · Omitted variable bias

True model `y = 1 + 1.0·x + 2.0·z + noise`, with `x = z + noise`. What will the coefficient on x be if z is not controlled for? Compute it with the formula first, then simulate.

```python
rng = np.random.default_rng(12)
w = pd.DataFrame({"z": rng.normal(size=5000)})
w["x"] = w["z"] + rng.normal(size=5000)
w["y"] = 1 + 1.0 * w["x"] + 2.0 * w["z"] + rng.normal(size=5000)

print(round(smf.ols("y ~ x + z", data=w).fit().params["x"], 3))
print(round(smf.ols("y ~ x", data=w).fit().params["x"], 3))
```

**Result:**

```text
1.001
2.002
```

> Bias = γ × δ. γ = 2 (the effect of z on y); δ = Cov(z, x)/Var(x) = 1/2. So without controlling for z the coefficient is about 1 + 2 × 0.5 = 2.

### Problem 11 · Adding a variable unrelated to x that affects y

Compare the coefficient and standard error of x1 in `y ~ x1` and `y ~ x1 + x2` (x2 is independent of x1 but affects y).

```python
short = smf.ols("y ~ x1", data=df).fit()
long = smf.ols("y ~ x1 + x2", data=df).fit()
print(round(short.params["x1"], 4), round(long.params["x1"], 4))
print(round(short.bse["x1"], 4), round(long.bse["x1"], 4))
```

**Result:**

```text
1.5197 1.5225
0.0872 0.0799
```

> The coefficient barely changes (x2 is uncorrelated with x1, so leaving it out causes no omitted variable bias), but the standard error shrinks: x2 explains away part of the noise in y. This is exactly the principle behind covariate adjustment (CUPED) in A/B tests.

### Problem 12 · Log–log model 🗣

The coefficient in `log(sales) ~ log(price)` is −1.3. How do you explain it to a business stakeholder?

**Reference answer:**

"Every 1% increase in price lowers sales by about 1.3%." The coefficient is the price elasticity. An absolute value above 1 means demand is elastic: the percentage drop in sales exceeds the percentage price increase, so total revenue falls.

Add two caveats: (1) this is an average relationship "holding everything else constant"; (2) if price was not set randomly (e.g. prices go up in peak season), it may be correlated with demand, so the coefficient is not necessarily causal.

### Problem 13 · The dummy variable trap 🗣

For a variable with 3 categories, why include only 2 dummy variables? What happens if you include all 3?

**Reference answer:**

The 3 dummies always sum to 1, which is perfectly collinear with the intercept column; $X^\top X$ is not invertible and the coefficients have no unique solution. So one category is dropped as the **reference**, and the other coefficients are interpreted as "the difference from the reference category".

Changing the reference category does not change the fitted values or the model's overall conclusions, only the meaning of the coefficients. statsmodels' `C(group)`, like R, uses the first category as the reference by default.

### Problem 14 · What the main effect means with an interaction 🗣

In the model `y ~ x * z`, the coefficient on x is 0.2 and not significant. Can you say "x has no effect on y"?

**Reference answer:**

No. With an interaction term, the coefficient on x is only the effect of x **when z = 0**. The effect of x is $\beta_x + \beta_{xz} z$, which may be large at other values of z. If z = 0 is not even in the range of the data (say z is age), the coefficient on its own has no practical meaning.

A common fix: center z (subtract its mean), so the coefficient on x becomes "the effect of x at the average z", which is easier to interpret.

### Problem 15 · Adding a variable correlated with x that does not affect y 🗣

You add a variable w to the regression; it is highly correlated with x but has no effect on y itself. What happens to the coefficient and standard error of x?

**Reference answer:**

The coefficient stays unbiased (w's true coefficient is 0, so there is no omitted variable bias), but the **standard error of x grows**: x and w are collinear, x has less "unique" variation, and the estimate becomes less precise (the VIF rises).

Contrast this with the computation problem above: adding a variable unrelated to x that affects y makes the standard error smaller. So the rule for choosing controls is: variables that explain y and are not themselves outcomes of the treatment help; variables correlated only with x and not with y just add variance.

### Problem 16 · A U-shaped residual plot 🗣

The plot of residuals against fitted values is U-shaped. What does that tell you? What do you do?

**Reference answer:**

The model is missing a nonlinear relationship, violating the "correctly specified linear form" assumption; the conditional mean of the error is not 0, so the coefficients are biased.

Fixes: add a squared term or other polynomial terms (`I(x ** 2)`), take logs of x or y, use piecewise terms or splines, or switch to a model that can capture nonlinearity. Afterwards redraw the residual plot and check that it has become a cloud of points with no structure.

*Short version:* A U-shaped residual plot signals a missed nonlinearity, so I'd add a squared term, transform a variable, or use splines, then recheck the residuals.

### Problem 17 · Is a model with R² = 0.05 useful? 🗣

A regression has R² of only 0.05, but the coefficient you care about is highly significant. Is the model useful?

**Reference answer:**

It depends on the goal. **If the goal is to predict individual y**, R² = 0.05 means very little is explained and predictive power is weak. **If the goal is to estimate the effect of some variable**, a low R² is perfectly acceptable: individual behavior is inherently noisy (e.g. personal spending), and as long as zero conditional mean holds, the coefficient is still unbiased and can be estimated precisely with a large sample.

A/B tests are the extreme example: the treatment variable explains only a tiny share of the outcome's variance, yet the effect estimate is fully credible.

### Problem 18 · What does BLUE mean? 🗣

The Gauss–Markov theorem says OLS is BLUE. What do the four letters stand for? What assumptions are needed?

**Reference answer:**

Best Linear Unbiased Estimator: among all estimators that are **linear** (linear functions of y) and **unbiased**, OLS has the **smallest** variance.

Assumptions needed: a linear model, full rank X, zero conditional mean, and homoskedastic, uncorrelated errors. **Normality is not needed.**

Note the qualifiers: it is only the best among linear unbiased estimators; a biased estimator (e.g. ridge) can have smaller MSE; under heteroskedasticity OLS is no longer best, and WLS is more efficient.

### Problem 19 · Common reasons exogeneity breaks 🗣

Why does zero conditional mean, $E[\varepsilon \mid X] = 0$, often fail in real data?

**Reference answer:**

(1) **Omitted variables**: a variable affects both X and Y but is not in the model; (2) **reverse causality**: Y also affects X (e.g. ad spend and sales influence each other); (3) **sample selection**: you only observe people who meet certain conditions (e.g. you only see wages for people who found jobs); (4) **measurement error**: X is measured imprecisely, which pulls the coefficient toward 0.

All four bias the coefficients, and more data does not fix any of them. The fix comes from research design: randomized experiments, instrumental variables, difference-in-differences, etc.

### Problem 20 · How to compare which of two variables matters more 🗣

In a regression the coefficient on x1 is 3.2 and on x2 is 0.04. Can you say x1 matters more than x2?

**Reference answer:**

You can't compare them directly; a coefficient's size depends on units. If x2 is "income in dollars" and x1 is "age", the units are completely different.

Options: (1) standardize all variables and then compare coefficients (how much y changes per one-standard-deviation change); (2) see how much R² or prediction error worsens when a variable is dropped; (3) compare using "realistically achievable changes" from the business side (you can't change age, but you can adjust price). Also, when the variables are correlated, "importance" is not uniquely defined in the first place.

## Recap

| Task | Code |
|---|---|
| Fit | `smf.ols("y ~ x1 + x2 + C(g)", data=df).fit()` |
| Extract | `.params`, `.bse`, `.pvalues`, `.conf_int().loc["x1", 0]` |
| OLS by hand | `np.linalg.solve(X.T @ X, X.T @ y)`, add the intercept column yourself |
| R² / adjusted R² | `.rsquared` / `.rsquared_adj`, formula $1 - (1 - R^2)\frac{n-1}{n-p-1}$ |
| Predict | `model.predict(new_df)` |
| Interactions / logs | `y ~ x * z`, marginal effect $\beta_x + \beta_{xz}z$; `np.log(y) ~ x` ≈ percentage change |
| Robust standard errors | `.fit(cov_type="HC3")` |
| Collinearity | `variance_inflation_factor(X, j)`, VIF = 1/(1 − R²ⱼ) |
| Omitted variable bias | bias = γ × δ |